In [ ]:
# Import everything necessary
import pandas as pd
import networkx as nx
import numpy as np
from itertools import combinations
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Step 1: Load the dataset and create graph
def load_data_and_create_graph():
    # Format: each line represents an edge with two node IDs separated by space
    df = pd.read_csv('facebook_combined.txt', sep=' ', header=None, names=['source', 'destination'])
    G = nx.from_pandas_edgelist(df, 'source', 'destination')
    return df, G

# Step 2: Create feature dataframe with negative sampling
def create_feature_dataframe(df, G):
    # Existing edges (positive samples)
    positive_samples = df.copy()
    positive_samples['label'] = 1

    # Generate all possible node pairs
    all_nodes = list(G.nodes())
    all_possible_pairs = list(combinations(all_nodes, 2))

    # Convert existing edges to set for faster lookup
    existing_edges = set([tuple(sorted((u, v))) for u, v in zip(df['source'], df['destination'])])

    # Filter out existing edges and sample negative samples
    non_edges = [pair for pair in all_possible_pairs if tuple(sorted(pair)) not in existing_edges]
    negative_indices = np.random.choice(len(non_edges), size=len(positive_samples), replace=False)
    negative_samples = [non_edges[i] for i in negative_indices]
    negative_samples = pd.DataFrame(negative_samples, columns=['source', 'destination'])
    negative_samples['label'] = 0

    # Combine positive and negative samples
    all_samples = pd.concat([positive_samples, negative_samples], ignore_index=True)

    # Calculate features for all pairs
    features = []
    for _, row in all_samples.iterrows():
        u, v = row['source'], row['destination']

        # Common neighbors
        cn = len(list(nx.common_neighbors(G, u, v)))

        # Jaccard coefficient
        jc = list(nx.jaccard_coefficient(G, [(u, v)]))[0][2]

        # Adamic-Adar index
        aa = list(nx.adamic_adar_index(G, [(u, v)]))[0][2]

        # Preferential attachment
        pa = list(nx.preferential_attachment(G, [(u, v)]))[0][2]

        features.append([cn, jc, aa, pa])

    # Add features to dataframe
    feature_cols = ['common_neighbors', 'jaccard_coeff', 'adamic_adar', 'pref_attachment']
    all_samples[feature_cols] = features

    return all_samples

# Step 3: Shuffle and print dataframe
def shuffle_and_print(df):
    shuffled_df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    print("First 10 rows of shuffled dataframe:")
    print(shuffled_df.head(10))
    return shuffled_df

# Step 4: Split data into train and test sets
def split_data(df):
    X = df[['common_neighbors', 'jaccard_coeff', 'adamic_adar', 'pref_attachment']]
    y = df['label']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    return X_train, X_test, y_train, y_test

# Step 5 & 6: Build logistic regression model with 5-fold CV
def cross_validate_model(X_train, y_train):
    model = LogisticRegression(max_iter=1000, random_state=42)
    scoring = ['accuracy', 'precision', 'recall', 'f1']
    cv_results = cross_validate(model, X_train, y_train, cv=5, scoring=scoring)

    print("\nCross-validation results:")
    print(f"Average accuracy: {np.mean(cv_results['test_accuracy']):.4f}")
    print(f"Average precision: {np.mean(cv_results['test_precision']):.4f}")
    print(f"Average recall: {np.mean(cv_results['test_recall']):.4f}")
    print(f"Average F1-score: {np.mean(cv_results['test_f1']):.4f}")

    return model

# Step 7: Final model evaluation
def evaluate_model(model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    print("\nTest set evaluation:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred):.4f}")
    print(f"Recall: {recall_score(y_test, y_pred):.4f}")
    print(f"F1-score: {f1_score(y_test, y_pred):.4f}")

# Main execution
def main():
    # Step 1
    print("Loading data and creating graph...")
    df, G = load_data_and_create_graph()

    # Step 2
    print("Creating feature dataframe with negative sampling...")
    feature_df = create_feature_dataframe(df, G)

    # Step 3
    print("Shuffling dataframe...")
    shuffled_df = shuffle_and_print(feature_df)

    # Step 4
    print("Splitting data into train and test sets...")
    X_train, X_test, y_train, y_test = split_data(shuffled_df)

    # Step 5 & 6
    print("Performing cross-validation...")
    model = cross_validate_model(X_train, y_train)

    # Step 7
    print("Evaluating final model...")
    evaluate_model(model, X_train, y_train, X_test, y_test)

if __name__ == "__main__":
    main()

Loading data and creating graph...
Creating feature dataframe with negative sampling...
Shuffling dataframe...
First 10 rows of shuffled dataframe:
   source  destination  label  common_neighbors  jaccard_coeff  adamic_adar  \
0     507          514      1              40.0       0.449438     9.774970   
1    3051         4018      0               0.0       0.000000     0.000000   
2    1020         1467      0               2.0       0.011111     0.332234   
3    1111         3935      0               0.0       0.000000     0.000000   
4     980         3129      0               0.0       0.000000     0.000000   
5     222          240      1               1.0       0.076923     0.170960   
6    3120         3146      1              42.0       0.466667     9.654676   
7    3667         3968      1              19.0       0.380000     5.418319   
8    1810         1839      1              73.0       0.376289    14.774402   
9    3245         2693      0               3.0       0.103448